### Data Preprocessing & Exploratory Data Analysis (EDA)

#### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

#### Data Loading

In [ ]:
df = pd.read_csv("product_info.csv")

print("Initial shape:", df.shape)
df.head()

#### Data Quality Analyzing

In [ ]:
missing_percent = df.isnull().mean().sort_values(ascending=False) * 100

missing_df = pd.DataFrame({
    "column": missing_percent.index,
    "missing_percent": missing_percent.values
})

missing_df

#### Handling Missing Value

Remove columns with excessive missing data (>60%) and rows where the Target Variable (rating) is missing, as these cannot be used for training.

In [ ]:
# Drop columns with more than 60% missing values
high_null_cols = missing_percent[missing_percent > 60].index
df = df.drop(columns=high_null_cols)

print("After dropping high-null columns:", df.shape)

In [ ]:
# Target variable
TARGET = "rating"

before = df.shape[0]
df = df.dropna(subset=[TARGET])
after = df.shape[0]

print(f"Rows removed due to missing target: {before - after}")
print("After target cleaning:", df.shape)

#### Numerical Imputation

Fill missing numerical values. We use specific logic for child_count (0 variations) and the median for other continuous variables.

In [ ]:
# If child_count is NaN, it implies the product has NO variations (0 children).

if 'child_count' in df.columns:
    df['child_count'] = df['child_count'].fillna(0)

num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

In [ ]:
# Fill remaining numeric nulls with Median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
# no. of rows and colums
print("Rows, Columns:", df.shape) 

print("\nColumn Name and Missing values per column:")
display(df.isna().sum())

#### Display Categorical columns

In [ ]:
pd.set_option('display.max_rows', None)

categorical_cols = [
    "size",
    "variation_type",
    "variation_value",
    "ingredients",     
    "highlights",
    "secondary_category", 
    "tertiary_category",         
]

for col in categorical_cols:
    vc = df[col].value_counts()
    display(vc) 

#### Display Unique Values of Tertiary

In [ ]:
unique_values = df['tertiary_category'].unique()
print(unique_values)

In [ ]:
unique_values = df['product_name'].unique()
print(unique_values)

#### Feature Engineering (Text Mining)

A significant number of products have missing tertiary_category labels. We use a keyword dictionary to infer these categories from the product_name, recovering data that would otherwise be deleted.

In [ ]:
category_keywords = {
    # ── FRAGRANCE ─────────────────────────
    "perfume": "Perfume",
    "eau de parfum": "Perfume",
    "eau de toilette": "Perfume",
    "cologne": "Cologne",
    "fragrance": "Perfume",
    "scent": "Perfume",
    "diffuser": "Home Scents", # Added from your list
    "candle": "Candles",       # Added from your list
    "aroma": "Home Scents",    # Added from your list

    # ── NAILS (New Section) ───────────────
    "nail polish": "Nail Polish",
    "gel nail": "Nail Polish",
    "top coat": "Nail Care",
    "base coat": "Nail Care",
    "nail care": "Nail Care",
    "nail treatment": "Nail Care",
    "cuticle": "Nail Care",
    "press-on": "False Nails",
    "artificial nail": "False Nails",
    "nail file": "Manicure & Pedicure Tools",
    "clipper": "Manicure & Pedicure Tools",
    "manicure": "Manicure & Pedicure Tools",
    "pedicure": "Manicure & Pedicure Tools",
    "remover": "Nail Polish Remover", # Careful, might catch makeup remover if not ordered right
    
    # ── TANNING (New Section) ─────────────
    "tanning": "Sunless Tanning",
    "self tan": "Sunless Tanning",
    "bronzing mousse": "Sunless Tanning",
    "tan tonic": "Sunless Tanning",
    "glow drops": "Sunless Tanning",

    # ── HAIR CARE ─────────────────────────
    "shampoo": "Shampoo",
    "conditioner": "Conditioner", 
    "hair mask": "Hair Masks",
    "leave-in": "Leave-In Conditioner",
    "hair oil": "Hair Oil",
    "hair serum": "Hair Serums",
    "scalp": "Scalp Treatments",
    "dry shampoo": "Dry Shampoo",
    "texture spray": "Hair Styling Products", # Added
    "volumizer": "Hair Styling Products",     # Added
    "hair gel": "Hair Styling Products",      # Added
    "hairspray": "Hair Spray",
    "hair spray": "Hair Spray",
    "heat protect": "Hair Styling Products",
    "hair dryer": "Hair Dryers",
    "straightener": "Hair Straighteners & Flat Irons",
    "flat iron": "Hair Straighteners & Flat Irons",
    "curling iron": "Curling Irons",
    "hair loss": "Hair Thinning & Hair Loss",
    "root touch-up": "Hair Dye & Root Touch-Ups",
    "hair dye": "Hair Dye & Root Touch-Ups",
    "hair tools": "Hair Styling Tools",       # Added

    # ── SKINCARE ──────────────────────────
    "serum": "Face Serums",
    "face serum": "Face Serums",
    "cleansing balm": "Face Wash & Cleansers", # Added specific
    "cleansing gel": "Face Wash & Cleansers",  # Added specific
    "cleansing foam": "Face Wash & Cleansers", # Added specific
    "cleanser": "Face Wash & Cleansers",
    "face wash": "Face Wash & Cleansers",
    "micellar water": "Face Wash & Cleansers", # Added
    "moisturizer": "Moisturizers",
    "cream": "Moisturizers",
    "lotion": "Moisturizers",
    "night cream": "Night Creams",
    "eye cream": "Eye Creams & Treatments",
    "eye treatment": "Eye Creams & Treatments",
    "toner": "Toners",
    "essence": "Mists & Essences",
    "facial spray": "Mists & Essences", # Added
    "mist": "Mists & Essences",
    "oil": "Face Oils",
    "facial oil": "Face Oils",
    "mask": "Face Masks",
    "sheet mask": "Sheet Masks",
    "peel": "Facial Peels",
    "exfoliat": "Exfoliators",
    "scrub": "Scrub & Exfoliants",
    "sunscreen": "Face Sunscreen",
    "spf": "Face Sunscreen",
    "acne": "Blemish & Acne Treatments",
    "blemish": "Blemish & Acne Treatments",
    "dark spot": "Treatments (Face)", # Added
    "retinol": "Treatments (Face)",   # Added
    "collagen": "Treatments (Face)",  # Added
    "anti-aging": "Anti-Aging",

    # ── MAKEUP ────────────────────────────
    "foundation": "Foundation",
    "tint": "Tinted Moisturizer", # Added
    "bb cream": "BB & CC Cream",
    "cc cream": "BB & CC Cream",
    "concealer": "Concealer",
    "color correct": "Color Correct",
    "setting spray": "Setting Spray & Powder",
    "primer": "Face Primer",
    "highlighter": "Highlighter",
    "bronzer": "Bronzer",
    "blush": "Blush",
    "contour": "Contour",
    "eyeshadow": "Eyeshadow",
    "eye palette": "Eye Palettes",  # Added
    "face palette": "Face Palettes", # Added
    "palette": "Makeup Palettes",   # Fallback
    "eyeliner": "Eyeliner",
    "mascara": "Mascara",
    "false lash": "False Eyelashes",
    "lash": "False Eyelashes",
    "lipstick": "Lipstick",
    "liquid lipstick": "Liquid Lipstick",
    "lip gloss": "Lip Gloss",
    "lip balm": "Lip Balm & Treatment",
    "lip butter": "Lip Balm & Treatment", # Added
    "lip treatment": "Lip Balm & Treatment", # Added
    "lip liner": "Lip Liner",
    "lip plumper": "Lip Plumper",
    "lip serum": "Lip Balm & Treatment",
    "lippe balm": "Lippe Balm",
    "brow": "Eyebrow", # Catches 'brow pencil', 'brow gel'

    # ── BODY CARE ─────────────────────────
    "body wash": "Body Wash & Shower Gel",
    "shower gel": "Body Wash & Shower Gel",
    "body lotion": "Body Lotions & Body Oils",
    "body oil": "Body Lotions & Body Oils",
    "body butter": "Body Lotions & Body Oils", # Added
    "hand cream": "Hand Cream & Foot Cream",
    "hand salve": "Hand Cream & Foot Cream",   # Added
    "foot cream": "Hand Cream & Foot Cream",
    "foot file": "Bath & Body Tools",          # Added
    "deodorant": "Deodorant & Antiperspirant",
    "antiperspirant": "Deodorant & Antiperspirant",
    "body mist": "Body Mist & Hair Mist",
    "hair mist": "Body Mist & Hair Mist",
    "body sunscreen": "Body Sunscreen",
    "cellulite": "Cellulite & Stretch Marks",

    # ── TOOLS & ACCESSORIES ───────────────
    "brush": "Brushes & Combs",
    "comb": "Brushes & Combs",
    "sponge": "Sponges & Applicators",
    "applicator": "Sponges & Applicators",
    "tweezer": "Tweezers & Eyebrow Tools",
    "eyelash curler": "Eyelash Curlers",
    "sharpen": "Sharpeners",
    "makeup bag": "Makeup Bags & Travel Cases",
    "microneedling": "Beauty Tools", # Added
    "roller": "Beauty Tools",        # Added
    "wand": "Beauty Tools",          # Added
    "device": "Beauty Tools",        # Added

    # ── SETS & KITS (Catch-all for Gifts) ─
    "hair set": "Hair Care Sets",
    "skincare set": "Skincare Sets",
    "makeup set": "Makeup Sets",
    "gift set": "Value & Gift Sets",
    "travel kit": "Value & Gift Sets",
    "discovery set": "Value & Gift Sets",
    "starter kit": "Value & Gift Sets",
    "duo": "Value & Gift Sets",
    "trio": "Value & Gift Sets",
    "kit": "Value & Gift Sets",      # Generic fallback
    "set": "Value & Gift Sets",      # Generic fallback

    # ── MEN'S GROOMING ────────────────────
    "aftershave": "Aftershave",
    "shave": "Shaving",

    # ── LIFESTYLE / OTHER ─────────────────
    "supplement": "Beauty Supplements",
    "vitamin": "Beauty Supplements",
    "holistic": "Holistic Wellness",
    "teeth whitening": "Teeth Whitening",
    "gift card": "Gift Cards",       # Added
    "google nest": "Lifestyle Tech", # Added for that one Google item
}

#### Data Transformation

In [ ]:
# Sort keywords by length to match longest phrases first (e.g., match "liquid lipstick" before "lipstick")
category_keywords = dict(
    sorted(category_keywords.items(), key=lambda x: len(x[0]), reverse=True)
)

# Define inference function
def infer_category(product_name):
    if pd.isna(product_name):
        return np.nan
    
    name = product_name.lower()
    
    for keyword, category in category_keywords.items():
        if keyword in name:
            return category
    
    return np.nan

#### Analyzing Missing Values of Before and After

In [ ]:
missing_before = df["tertiary_category"].isna().sum()
print("Missing before:", missing_before)

df.loc[df["tertiary_category"].isna(), "tertiary_category"] = (
    df.loc[df["tertiary_category"].isna(), "product_name"]
    .apply(infer_category)
)

missing_after = df["tertiary_category"].isna().sum()
print("Missing after:", missing_after)


#### Removing unused Columns

In [ ]:
# Create a list of the columns you want to remove
unnecessary_columns = [
    "variation_type", 
    "variation_value", 
    "ingredients", 
    "highlights"
]

# Drop them from the dataframe
# errors='ignore' ensures the code won't crash if one of these columns is already missing
df = df.drop(columns=unnecessary_columns, errors='ignore')

# Check the result
print(df.columns)

### Dropping Null Rows

In [ ]:
df = df.dropna(subset=["tertiary_category","secondary_category","size"])

#### Used columns in this Project

In [ ]:
# no. of rows and colums
print("Rows, Columns:", df.shape) 

print("\nColumn Name and Missing values per column:")
display(df.isna().sum())

#### Standardize Text

In [ ]:
# Clean up string columns
text_cols = ["product_name","brand_name", "primary_category", "secondary_category", "tertiary_category"]

for col in text_cols:
    if col in df.columns:
        # Remove empty spaces at start/end and capitalize consistent
        df[col] = df[col].str.strip().str.title()

#### Data Export

In [ ]:
df.to_csv('cleaned_product_info.csv', index=False)

print("File saved successfully as 'cleaned_product_info.csv'")

#### Exploratory Data Analysis (Visualization)

1. The J-Curve (Rating Distribution)
2. Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set the style for professional-looking plots
sns.set_style("whitegrid")

# ==========================================
# PLOT 1: The "J-Curve" (Rating Distribution)
# ==========================================
plt.figure(figsize=(10, 6))

# Create the histogram
sns.histplot(df['rating'], bins=20, kde=True, color='#CE2029') # Sephora Red color

# Add vertical lines to show your Justification
plt.axvline(df['rating'].mean(), color='black', linestyle='--', linewidth=2, label=f"Mean Rating ({df['rating'].mean():.2f})")
plt.axvline(3.0, color='blue', linestyle=':', linewidth=2, label="Low/Med Threshold (3.0)")
plt.axvline(4.0, color='green', linestyle=':', linewidth=2, label="Med/High Threshold (4.0)")

# Labels and Title
plt.title("Distribution of Product Ratings (The 'J-Curve')", fontsize=16, fontweight='bold')
plt.xlabel("Star Rating", fontsize=12)
plt.ylabel("Number of Products", fontsize=12)
plt.legend()

# Save the plot
plt.savefig("J_Curve_Histogram.png", dpi=300)
plt.show()

print("Plot 1 Note: Notice how the peak is heavily skewed to the right (5.0).")
print("This proves that a 'mathematical average' (2.5) would be a bad threshold.")

# ==========================================
# PLOT 2: Correlation Heatmap
# ==========================================
plt.figure(figsize=(12, 10))

# Select only numerical columns for correlation
# We exclude 'id' columns because they are random numbers, not features
numerical_cols = ['rating', 'price_usd', 'loves_count', 'reviews']

# Calculate correlation matrix
corr_matrix = df[numerical_cols].corr()

# Create the heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # Optional: Mask the upper triangle to make it cleaner
sns.heatmap(corr_matrix, 
            annot=True,       # Show the numbers
            fmt=".2f",        # 2 decimal places
            cmap='coolwarm',  # Red=High Correl, Blue=Low Correl
            mask=mask,        # Hide duplicate top half
            linewidths=0.5)

plt.title("Feature Correlation Matrix", fontsize=16, fontweight='bold')

# Save the plot
plt.savefig("Correlation_Heatmap.png", dpi=300)
plt.show()

print("Plot 2 Note: Look at the 'rating' row.")
print("If 'price_usd' has a positive number (e.g., 0.15), it means expensive items tend to have higher ratings.")